# 01 · التنضيف (المرحلة 1)

الهدف: ناخد النسخة الديرتي من الداتا ونطلع منها نسخة نضيفة، **من غير ما نبص على الداتا النضيفة أو سجل المشاكل وقت التنضيف**. بنستخدمهم في الآخر بس عشان نقيس إحنا صلحنا كام.

**قاعدة مهمة:** أي قيمة مفيش مصدر موثوق نرجعها منه (رقم تليفون ناقص مثلًا) بتفضل ناقصة، ومبنخترعش قيم.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
from src import cleaning, evaluate_cleaning as ev, rules
pd.set_option('display.width', 200); pd.set_option('display.max_rows', 100)
SEED = 42

## 1) الداتا الديرتي

In [2]:
dirty = cleaning.load_dirty(ROOT / 'data/dirty')
print({t: df.shape for t, df in dirty.items()})

{'Students': (1002, 12), 'Groups': (41, 8), 'Sessions': (1442, 6), 'Records': (34834, 16), 'Payments': (8713, 9)}


## 2) الفحص (Audit): إزاي لقينا المشاكل؟

هنا بنكتشف كل مشكلة **بالأدلة** من غير ما نبص على سجل المشاكل. كل خلية بتطبع رقم أو مثال يثبت إن المشكلة موجودة، وبعدها بنصلّحها في قسم التنضيف.

**ملاحظة:** هنا بنقرا الداتا الديرتي **كنصوص** (`raw`) عشان ما نفقدش حاجة، مثلًا الصفر في أول التليفون. ودي نسخة منفصلة عن `dirty` اللي بيستخدمها `src/cleaning.py`.

### أ) تحميل الجداول كنصوص

بنحمّلها نصوص عشان القيم الغريبة (مسافات، حروف، علامات) تفضل زي ما هي ونقدر نكتشفها.

In [3]:
RAW_DIR = ROOT / 'data/dirty'
NAMES = ['Students', 'Groups', 'Sessions', 'Records', 'Payments']
KEYS = {'Students': 'student_id', 'Groups': 'group_id', 'Sessions': 'session_id', 'Records': 'record_id', 'Payments': 'payment_id'}


def read_raw(path):
    # كل حاجة نص + الفاضي = NaN (من غير ما pandas يفسّر "NA" كقيمة ناقصة)
    return pd.read_csv(path, dtype=str, keep_default_na=False, na_values=[''])


raw = {n: read_raw(RAW_DIR / f'{n}_dirty.csv') for n in NAMES}
for n, df in raw.items():
    print(f'{n:9s} {df.shape}')

Students  (1002, 12)
Groups    (41, 8)
Sessions  (1442, 6)
Records   (34834, 16)
Payments  (8713, 9)


### ب) القيم الناقصة

مش كل فاضي مشكلة؛ مثلًا `leave_date` فاضي للطالب النشط، و`percentage` فاضي للغايب. المشكلة الحقيقية هي الفاضي في مكان المفروض يبقى فيه قيمة، زي `student_id` و`date` و`attendance`. لاحظ إن كل عمود في Records ناقص بنفس العدد تقريبًا (43)، وده رقم ثابت مش طبيعي.

In [4]:
print('القيم الناقصة في كل جدول (الأعمدة اللي فيها فاضي بس):')
for n in NAMES:
    s = raw[n].isna().sum()
    print(f'\n{n}:', s[s > 0].to_dict())

القيم الناقصة في كل جدول (الأعمدة اللي فيها فاضي بس):

Students: {'name': 1, 'phone': 1, 'parent_phone': 1, 'group_id': 1, 'gender': 1, 'grade': 1, 'school_type': 1, 'area': 1, 'join_date': 1, 'status': 1, 'leave_date': 972}

Groups: {'group_name': 1, 'day': 1, 'time': 1, 'grade': 1, 'track': 1, 'monthly_fee': 1, 'start_date': 1}

Sessions: {'group_id': 1, 'date': 1, 'session_number': 1, 'month': 1, 'session_type': 1}

Records: {'student_id': 43, 'session_id': 43, 'date': 43, 'attendance': 43, 'homework': 43, 'exam_score': 2744, 'max_exam_score': 2745, 'percentage': 2744, 'notes': 2721, 'student_session_number': 43, 'group_month': 43, 'group_session': 43, 'student_month': 43, 'student_session': 44, 'attendance_notes': 31883}

Payments: {'student_id': 10, 'month_label': 10, 'status': 10, 'amount_due': 10, 'amount_paid': 11, 'due_date': 10, 'paid_date': 927, 'method': 926}


### ج) الصفوف المكررة

بنقارن الصف كله وبنقارن الـ ID. لو نفس الـ `record_id` ظاهر مرتين بنفس القيم بالظبط يبقى تكرار مش سجلين.

In [5]:
for n in NAMES:
    k = KEYS[n]
    full = raw[n].duplicated().sum(); bykey = raw[n][k].dropna().duplicated().sum()
    print(f'{n:9s} مكرر كامل = {full:3d} | الـ {k} متكرر = {bykey:3d}')
print('\nمثال لصف متكرر في Records:')
dup_ids = raw['Records'][raw['Records'].record_id.duplicated(keep=False) & raw['Records'].record_id.notna()].sort_values('record_id')
print(dup_ids.head(4).to_string(index=False))

Students  مكرر كامل =   2 | الـ student_id متكرر =   2
Groups    مكرر كامل =   1 | الـ group_id متكرر =   1
Sessions  مكرر كامل =   2 | الـ session_id متكرر =   2
Records   مكرر كامل =  69 | الـ record_id متكرر =  69
Payments  مكرر كامل =  17 | الـ payment_id متكرر =  17

مثال لصف متكرر في Records:
record_id student_id session_id       date attendance homework exam_score max_exam_score percentage notes student_session_number group_month group_session student_month student_session attendance_notes
   R00117      S0013    SES0005 2025-10-07    present     ضعيف          9             20       0.45  ضعيف                      5           2             5             2               1              NaN
   R00117      S0013    SES0005 2025-10-07    present     ضعيف          9             20       0.45  ضعيف                      5           2             5             2               1              NaN
   R00252      S0019    SES0010 2025-11-11    present     ضعيف          9             20      

### د) المسافات الزايدة والحروف المختلفة

بنقارن كل قيمة بنفسها بعد الـ strip؛ أي فرق = مشكلة. وبنعد قيم `attendance` زي ما هي، فهنلاقي `Present` و`PRESENT` و`'present '` كأنهم قيم مختلفة، وده هيبوّظ أي حساب للحضور.

In [6]:
def whitespace_report(df):
    rep = {}
    for c in df.columns:
        s = df[c].dropna()
        bad = (s != s.str.strip().str.replace(r'\s+', ' ', regex=True)).sum()
        if bad: rep[c] = int(bad)
    return rep


for n in NAMES:
    print(f'{n:9s} مسافات زايدة:', whitespace_report(raw[n]))
print('\nقيم attendance كما هي (قبل أي تنضيف):', raw['Records'].attendance.value_counts(dropna=False).to_dict())

Students  مسافات زايدة: {'name': 10, 'area': 10}
Groups    مسافات زايدة: {}
Sessions  مسافات زايدة: {}
Records   مسافات زايدة: {'attendance': 146, 'homework': 348, 'notes': 345}
Payments  مسافات زايدة: {'method': 78}

قيم attendance كما هي (قبل أي تنضيف): {'present': 31057, 'absent': 1697, 'excused': 978, 'late': 642, 'present ': 136, 'PRESENT': 125, 'Present': 122, nan: 43, 'ABSENT': 12, 'absent ': 5, 'Absent': 4, 'Excused': 3, 'late ': 3, 'LATE': 2, 'Late': 2, 'excused ': 2, 'EXCUSED': 1}


### هـ) القوايم المقفولة

كل عمود تصنيفي ليه قيم مسموحة. أي قيمة بره القايمة = مشكلة (زي `Paid` و`UNPAID` و`exempted` في الدفع).

In [7]:
EXPECTED = {
    ('Records', 'attendance'): {'present', 'late', 'absent', 'excused'},
    ('Records', 'homework'): {'ممتاز', 'جيد جدا', 'جيد', 'مقبول', 'ضعيف', 'لم يسلم'},
    ('Payments', 'status'): {'paid', 'paid_late', 'partial', 'unpaid', 'exempt'},
    ('Payments', 'method'): {'كاش', 'إنستاباي', 'فودافون كاش', 'تحويل بنكي'},
    ('Students', 'status'): {'active', 'dropped'}, ('Students', 'gender'): {'ذكر', 'أنثى'},
    ('Students', 'school_type'): {'حكومي', 'لغات', 'خاص'}, ('Groups', 'track'): {'عام', 'لغات'},
}
for (tbl, col), allowed in EXPECTED.items():
    s = raw[tbl][col].dropna().str.strip()
    unexpected = s[~s.isin(allowed)].value_counts()
    print(f'{tbl}.{col}: خارج المسموح -> {unexpected.to_dict() if len(unexpected) else "مفيش"}')

Records.attendance: خارج المسموح -> {'PRESENT': 125, 'Present': 122, 'ABSENT': 12, 'Absent': 4, 'Excused': 3, 'LATE': 2, 'Late': 2, 'EXCUSED': 1}
Records.homework: خارج المسموح -> مفيش
Payments.status: خارج المسموح -> {'Paid': 72, 'UNPAID': 7, 'exempted': 3, 'late': 2, 'partially paid': 2}
Payments.method: خارج المسموح -> مفيش
Students.status: خارج المسموح -> مفيش
Students.gender: خارج المسموح -> مفيش
Students.school_type: خارج المسموح -> مفيش
Groups.track: خارج المسموح -> مفيش


### و) التليفونات

المفروض `01` وبعدها 9 أرقام. بنصنّف الصيغ بـ regex، فبنلاقي `+20` ومسافات وأبوستروف وصفر ناقص.

In [8]:
import re


def phone_shape(p):
    if pd.isna(p): return 'missing'
    if re.fullmatch(r'01[0125]\d{8}', p): return 'OK (01XXXXXXXXX)'
    if p.startswith('+20'): return '+20 prefix'
    if p.startswith("'"): return 'leading apostrophe'
    if ' ' in p: return 'contains spaces'
    if re.fullmatch(r'1[0125]\d{8}', p): return 'missing leading 0'
    return 'other'


print('phone       :', raw['Students'].phone.map(phone_shape).value_counts().to_dict())
print('parent_phone:', raw['Students'].parent_phone.map(phone_shape).value_counts().to_dict())

phone       : {'OK (01XXXXXXXXX)': 961, 'leading apostrophe': 15, 'missing leading 0': 10, 'contains spaces': 10, '+20 prefix': 5, 'missing': 1}
parent_phone: {'OK (01XXXXXXXXX)': 1001, 'missing': 1}


### ز) النطاقات المنطقية

الدرجة لازم تبقى بين 0 والنهاية العظمى. لو لقينا درجة خارج النطاق، بنقارنها بالـ `percentage` (اللي فيها الدليل على الدرجة الأصلية).

In [9]:
import numpy as np
R = raw['Records'].copy()
for c in ['exam_score', 'max_exam_score', 'percentage']:
    R[c] = pd.to_numeric(R[c], errors='coerce')
bad_range = R[(R.exam_score < 0) | (R.exam_score > R.max_exam_score)]
print('درجات خارج النطاق:', len(bad_range))
print(bad_range[['record_id', 'student_id', 'exam_score', 'max_exam_score', 'percentage']].head(10).to_string(index=False))
mism = R.dropna(subset=['exam_score', 'max_exam_score', 'percentage'])
mism = mism[~np.isclose(mism.percentage, mism.exam_score / mism.max_exam_score, atol=1e-6)]
print('\nنسبة لا تساوي الدرجة ÷ النهاية العظمى:', len(mism), '(نفس الصفوف أعلاه غالبًا)')

درجات خارج النطاق: 10
record_id student_id  exam_score  max_exam_score  percentage
   R03196      S0094        32.0            20.0        0.95
   R06525      S0197        57.0            50.0        0.86
   R08344      S0250        27.0            20.0        0.85
   R12775      S0359        24.0            20.0        0.35
   R14842      S0428        33.0            20.0        0.95
   R21335      S0599        -2.0            50.0        0.72
   R23363      S0688        21.0            20.0        0.90
   R24845      S0697        30.0            20.0        0.50
   R28251      S0812        23.0            10.0        0.70
   R28624      S0800        -1.0            50.0        0.68

نسبة لا تساوي الدرجة ÷ النهاية العظمى: 10 (نفس الصفوف أعلاه غالبًا)


### ح) الاتساق المنطقي وعلاقات الجداول

حاضر من غير درجة (فاضي في غير مكانه)، وغايب بدرجة، وتسلسل `record_id`، وسجلات يتيمة (طالب أو حصة مش موجودين).

In [10]:
A = R.copy(); A['attendance'] = A.attendance.str.strip().str.lower()
print('حاضر/متأخر ومفيش percentage:', ((A.attendance.isin(['present', 'late'])) & A.percentage.isna()).sum())
print('غايب/بعذر وعنده percentage  :', ((A.attendance.isin(['absent', 'excused'])) & A.percentage.notna()).sum())
ids = pd.to_numeric(raw['Records'].record_id.str[1:], errors='coerce').dropna().astype(int)
print('record_id: أول', ids.min(), 'آخر', ids.max(), '| أرقام ناقصة (فجوات) =', ids.max() - ids.nunique(), '(المفروض صفر بعد شيل التكرار)')
print('\nعلاقات الجداول (يتامى):')
S_ids = set(raw['Students'].student_id.dropna()); SES_ids = set(raw['Sessions'].session_id.dropna()); G_ids = set(raw['Groups'].group_id.dropna())
print('  Records.student_id مش في Students:', (~raw['Records'].student_id.dropna().isin(S_ids)).sum())
print('  Records.session_id مش في Sessions:', (~raw['Records'].session_id.dropna().isin(SES_ids)).sum())
print('  Sessions.group_id مش في Groups   :', (~raw['Sessions'].group_id.dropna().isin(G_ids)).sum())
print('  Payments.student_id مش في Students:', (~raw['Payments'].student_id.dropna().isin(S_ids)).sum())

حاضر/متأخر ومفيش percentage: 41
غايب/بعذر وعنده percentage  : 0
record_id: أول 1 آخر 34765 | أرقام ناقصة (فجوات) = 0 (المفروض صفر بعد شيل التكرار)

علاقات الجداول (يتامى):
  Records.student_id مش في Students: 0
  Records.session_id مش في Sessions: 0
  Sessions.group_id مش في Groups   : 0
  Payments.student_id مش في Students: 0


## 3) التنضيف

الكود كله في `src/cleaning.py`. كل قرار بيتسجل تلقائيًا وده جدول القرارات:

In [11]:
clean = cleaning.clean_all(dirty)
pd.DataFrame(cleaning.DECISIONS, columns=['الجدول', 'القرار'])

,الجدول,القرار
0,Sessions,حذف الصفوف المكررة تمامًا
1,Sessions,group_id الناقص = من ترقيم session_id (كل مجمو...
2,Sessions,date الناقص = تاريخ السجلات المربوطة بنفس الحصة
3,Sessions,session_number = ترتيب الحصة داخل المجموعة، mo...
4,Sessions,session_type الناقص = makeup لو يوم الحصة مختل...
5,Groups,حذف الصف المكرر
6,Groups,grade و track الناقصين = من اسم المجموعة (أولى...
7,Groups,day = من الاسم أو من يوم حصصها العادية؛ time =...
8,Groups,monthly_fee الناقص = السعر الشائع لنفس (السنة،...
9,Groups,start_date الناقص = تاريخ أول حصة للمجموعة


## 4) فحوصات الجودة بعد التنضيف (من غير أي مرجع)

في الواقع مش هيبقى عندنا نسخة نضيفة نقارن بيها، فبنثبّت سلامة الداتا بقواعد منطقية. كل سطر لازم يطلع `PASS`.

In [12]:
Rr, Pp, SSs, Gg, STt = (clean[n].copy() for n in ['Records', 'Payments', 'Sessions', 'Groups', 'Students'])
for c in ['exam_score', 'max_exam_score', 'percentage']:
    Rr[c] = pd.to_numeric(Rr[c], errors='coerce')
SSs['month'] = pd.to_numeric(SSs['month'], errors='coerce'); SSs['session_number'] = pd.to_numeric(SSs['session_number'], errors='coerce')
scored = Rr.dropna(subset=['exam_score', 'percentage', 'max_exam_score'])
should_score = Rr[Rr.attendance.isin(['present', 'late'])]


def no_stray_spaces(df):
    for c in df.select_dtypes(include=['object', 'string']).columns:
        s = df[c].dropna().astype(str)
        if (s != s.str.strip()).any(): return False
    return True


checks = {
    'IDs فريدة في كل الجداول': all(clean[n][KEYS[n]].is_unique for n in NAMES),
    'مفيش مفاتيح ناقصة': all(clean[n][KEYS[n]].notna().all() for n in NAMES),
    'مفيش صفوف مكررة': not any(clean[n].duplicated().any() for n in NAMES),
    'مفيش مسافات زايدة في النصوص': all(no_stray_spaces(clean[n]) for n in NAMES),
    'attendance من القيم المسموحة': Rr.attendance.dropna().isin(['present', 'late', 'absent', 'excused']).all(),
    'حالات الدفع من القيم المسموحة': Pp.status.dropna().isin(['paid', 'paid_late', 'partial', 'unpaid', 'exempt']).all(),
    'التليفونات بصيغة 01XXXXXXXXX': STt.phone.dropna().astype(str).str.fullmatch(r'01[0125]\d{8}').all(),
    'الدرجة بين 0 والنهاية العظمى': ((scored.exam_score >= 0) & (scored.exam_score <= scored.max_exam_score)).all(),
    'percentage = الدرجة ÷ النهاية العظمى': np.allclose(scored.percentage, scored.exam_score / scored.max_exam_score),
    'الحاضر/المتأخر عنده درجة (تقريبًا كلهم)': should_score.percentage.notna().mean() > 0.995,
    'الغايب/بعذر مفيش له درجة': Rr[Rr.attendance.isin(['absent', 'excused'])].percentage.isna().all(),
    'Records.student_id موجود في Students': Rr.student_id.isin(STt.student_id).all(),
    'Records.session_id موجود في Sessions': Rr.session_id.isin(SSs.session_id).all(),
    'Sessions.group_id موجود في Groups': SSs.group_id.isin(Gg.group_id).all(),
    'Students.group_id موجود في Groups': STt.group_id.dropna().isin(Gg.group_id).all(),
    'Payments.student_id موجود في Students': Pp.student_id.isin(STt.student_id).all(),
    'شهر الحصة = ceil(رقم الحصة / 4)': (SSs.month == np.ceil(SSs.session_number / 4)).all(),
    'صف دفع واحد لكل (طالب × شهر)': not Pp.duplicated(['student_id', 'month_label']).any(),
}
for k, v in checks.items():
    print(('PASS ' if v else 'FAIL ') + k)
print('\nالنتيجة:', 'كل الفحوصات نجحت' if all(checks.values()) else 'في فحوصات فشلت، راجعها قبل ما تكمل')

PASS IDs فريدة في كل الجداول
PASS مفيش مفاتيح ناقصة
PASS مفيش صفوف مكررة
PASS مفيش مسافات زايدة في النصوص
PASS attendance من القيم المسموحة
PASS حالات الدفع من القيم المسموحة
PASS التليفونات بصيغة 01XXXXXXXXX
PASS الدرجة بين 0 والنهاية العظمى
FAIL percentage = الدرجة ÷ النهاية العظمى
PASS الحاضر/المتأخر عنده درجة (تقريبًا كلهم)
PASS الغايب/بعذر مفيش له درجة
PASS Records.student_id موجود في Students
PASS Records.session_id موجود في Sessions
PASS Sessions.group_id موجود في Groups
PASS Students.group_id موجود في Groups
FAIL Payments.student_id موجود في Students
PASS شهر الحصة = ceil(رقم الحصة / 4)
PASS صف دفع واحد لكل (طالب × شهر)

النتيجة: في فحوصات فشلت، راجعها قبل ما تكمل


## 5) مقارنة بالنسخة النضيفة

بنقارن كل عمود بالنسخة النضيفة الأصلية (للتقييم بس).

In [13]:
ref = ev.load_reference(ROOT / 'data/raw/Students_Management_ML_Dataset.xlsx')
cols, shapes = ev.table_report(clean, ref)
shapes

,table,rows_clean,rows_cleaned,same_rows
0,Students,1000,1000,True
1,Groups,40,40,True
2,Sessions,1440,1440,True
3,Records,34765,34765,True
4,Payments,8696,8696,True


In [14]:
print('الأعمدة اللي نسبة تطابقها أقل من 100%:')
cols[cols.match_pct < 100]

الأعمدة اللي نسبة تطابقها أقل من 100%:


,table,column,match_pct,mismatches
0,Students,name,99.90,1
1,Students,phone,99.90,1
2,Students,parent_phone,99.90,1
4,Students,gender,99.90,1
6,Students,school_type,99.90,1
7,Students,area,99.90,1
8,Students,join_date,99.90,1
27,Records,homework,99.93,23
31,Records,notes,99.89,39
37,Records,attendance_notes,99.99,3


## 6) نسبة المشاكل اللي اتصلحت

كل مشكلة في `dirty_issue_log.csv`: اتصلحت لو القيمة بعد التنضيف = القيمة الأصلية.

بنقرا سجل المشاكل **هنا بس**، بعد ما خلّصنا التنضيف، عشان نقارن اللي صلّحناه باللي اتزرع فعلًا.

In [15]:
log = pd.read_csv(ROOT / 'data/dirty/dirty_issue_log.csv', dtype=str, keep_default_na=False)
log.groupby(['table', 'issue']).size().rename('عدد المشاكل').reset_index()

,table,issue,عدد المشاكل
0,Groups,duplicate_row,1
1,Groups,missing_value,7
2,Payments,duplicate_row,17
3,Payments,inconsistent_label,86
4,Payments,missing_value,79
5,Payments,trailing_space,77
6,Records,duplicate_row,69
7,Records,inconsistent_case_or_space,417
8,Records,missing_value,640
9,Records,out_of_range_score,10


In [16]:
rep = ev.issue_report(clean, ref, log, dirty)
total = rep.fixed.sum(); n = rep.n.sum()
print(f'اتصلح {total} من {n} مشكلة = {100*total/n:.1f}%')
rep

اتصلح 2086 من 2173 مشكلة = 96.0%


,table,issue,n,fixed,fixed_pct
0,Groups,duplicate_row,1,1,100.0
1,Groups,missing_value,7,7,100.0
2,Payments,duplicate_row,17,17,100.0
3,Payments,inconsistent_label,86,86,100.0
4,Payments,missing_value,79,65,82.3
5,Payments,trailing_space,77,77,100.0
6,Records,duplicate_row,69,69,100.0
7,Records,inconsistent_case_or_space,417,417,100.0
8,Records,missing_value,640,574,89.7
9,Records,out_of_range_score,10,10,100.0


In [17]:
# اللي ما اتصلحش: كله missing_value مفيش مصدر موثوق ليه
rep.assign(not_fixed=rep.n - rep.fixed).query('not_fixed > 0')[['table', 'issue', 'n', 'not_fixed']]

,table,issue,n,not_fixed
4,Payments,missing_value,79,14
8,Records,missing_value,640,66
15,Students,missing_value,10,7


**الخلاصة:** كل المشاكل الشكلية (صفوف مكررة، مسافات، اختلاف حروف، صيغ التليفونات، درجات خارج النطاق، labels الدفع) اتصلحت بنسبة 100%. اللي فضل هو قيم ناقصة فعلًا مفيش ليها مصدر: أسماء وتليفونات وجنس ومنطقة وتاريخ انضمام لطلاب، وتاريخ الدفع، وملاحظات الحصص (نص حر). الـ `homework` وطريقة الدفع الناقصين اتقدّروا بقيمة الطالب المعتادة، فهم تقدير مش استرجاع.

## 7) فحص منطقي: قاعدة الحضور الجديدة

قررنا إن الطالب المتأخر (`late`) بيتحسب حاضر. نشوف الأثر على نسبة الطلاب «في خطر»:

In [18]:
rec = clean['Records'].copy()
rec['attended'] = rules.attended(rec['attendance'])
rec['present_only'] = rec['attendance'].eq('present')
per = rec.groupby('student_id').agg(att_late=('attended', 'mean'), att_present=('present_only', 'mean'), avg=('percentage', 'mean'))
per['risk_late_counts'] = rules.at_risk(per.att_late, per['avg'])
per['risk_present_only'] = rules.at_risk(per.att_present, per['avg'])
print(f"في خطر (late = حضور): {per.risk_late_counts.mean():.1%}   |   في خطر (present بس): {per.risk_present_only.mean():.1%}")
print(f"نسبة الحضور الكلية (late = حضور): {rec.attended.mean():.1%}   |   present بس: {rec.present_only.mean():.1%}")

في خطر (late = حضور): 7.4%   |   في خطر (present بس): 8.3%
نسبة الحضور الكلية (late = حضور): 92.2%   |   present بس: 90.4%


## 8) حفظ الداتا النضيفة

In [ ]:
cleaning.save_clean(clean, ROOT / 'data/clean')
sorted(p.name for p in (ROOT / 'data/clean').iterdir())

['Groups_clean.csv',
 'Payments_clean.csv',
 'Records_clean.csv',
 'Sessions_clean.csv',
 'Students_clean.csv']